# Subset Sum with Grover Search — High-Level `WeightedAdder` Version

This notebook implements a **correct high-level Grover oracle** for Subset Sum using Qiskit's `WeightedAdder`.

This version is good for understanding the algorithmic idea:

\[
f(x)=1 \quad \Longleftrightarrow \quad \sum_i x_i a_i = T.
\]

It does **not** hard-code the solution.  
It does, however, hide the arithmetic inside Qiskit's `WeightedAdder`.

So this notebook answers:

> How do we solve a small Subset Sum instance with Grover search in Qiskit?

The companion paper-outline notebook answers:

> How does the paper explicitly build the oracle with selector bits, constant registers, shadow registers, running sums, comparison, phase kickback, and uncomputation?

## Problem instance

We use

\[
S=\{1,2,3\}, \qquad T=3.
\]

The valid subsets are:

\[
\{3\}, \qquad \{1,2\}.
\]

So the marked selector bitstrings in internal order \((x_0,x_1,x_2)\) are:

\[
(0,0,1), \qquad (1,1,0).
\]

In [ ]:
# If needed:
# !pip install qiskit qiskit-aer matplotlib pylatexenc

from math import pi, sqrt, floor
from itertools import product

from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.circuit.library import WeightedAdder
from qiskit_aer import AerSimulator

In [ ]:
S = [1, 2, 3]
T = 3
n = len(S)

solutions = []
for bits in product([0, 1], repeat=n):
    if sum(bit * a for bit, a in zip(bits, S)) == T:
        solutions.append(bits)

print("S =", S)
print("T =", T)
print("Classical solutions in x[0],x[1],x[2] order:", solutions)

## What `WeightedAdder` does

`WeightedAdder(weights=S)` constructs a reversible arithmetic circuit that computes:

\[
|x\rangle|0\rangle \mapsto |x\rangle\left|\sum_i x_i a_i\right\rangle.
\]

That is exactly the weighted-sum part needed for Subset Sum.

Then we compare the sum register with \(T\), phase-flip the matching states, and uncompute.

In [ ]:
def little_endian_bits(value: int, width: int):
    return [(value >> j) & 1 for j in range(width)]


def phase_flip_on_sum_equal_target(qc, sum_qubits, target):
    '''
    Apply phase -1 when the sum register equals target.

    This is a direct phase oracle:
        |sum> -> -|sum> if sum == target.
    '''
    target_bits = little_endian_bits(target, len(sum_qubits))

    # Convert target pattern to all-ones.
    for q, bit in zip(sum_qubits, target_bits):
        if bit == 0:
            qc.x(q)

    # Phase flip all-ones.
    if len(sum_qubits) == 1:
        qc.z(sum_qubits[0])
    else:
        qc.mcp(pi, sum_qubits[:-1], sum_qubits[-1])

    # Undo conversion.
    for q, bit in zip(sum_qubits, target_bits):
        if bit == 0:
            qc.x(q)


def subset_sum_oracle_weighted_adder(S, target):
    '''
    High-level subset-sum phase oracle using Qiskit's WeightedAdder.

    It computes sum_i x_i a_i, phase-flips if the sum equals target,
    then uncomputes the sum.
    '''
    n = len(S)
    adder = WeightedAdder(num_state_qubits=n, weights=S)

    qc = QuantumCircuit(adder.num_qubits, name="Oracle_WeightedAdder")

    # Compute weighted sum into adder's sum register.
    qc.append(adder.to_gate(), range(adder.num_qubits))

    # In WeightedAdder, the first n qubits are state qubits.
    # The next adder.num_sum_qubits are sum qubits.
    sum_start = n
    sum_qubits = list(range(sum_start, sum_start + adder.num_sum_qubits))

    phase_flip_on_sum_equal_target(qc, sum_qubits, target)

    # Uncompute weighted sum.
    qc.append(adder.inverse().to_gate(), range(adder.num_qubits))

    return qc.to_gate()

## Diffusion operator

Grover diffusion acts only on the selector bits \(x\), not the arithmetic workspace.

In [ ]:
def diffusion_on_first_n_qubits(n, total_qubits):
    qc = QuantumCircuit(total_qubits, name="Diffusion")

    xs = list(range(n))

    qc.h(xs)
    qc.x(xs)

    if n == 1:
        qc.z(xs[0])
    else:
        qc.h(xs[-1])
        qc.mcx(xs[:-1], xs[-1])
        qc.h(xs[-1])

    qc.x(xs)
    qc.h(xs)

    return qc.to_gate()

In [ ]:
oracle = subset_sum_oracle_weighted_adder(S, T)
total_qubits = oracle.num_qubits
diffusion = diffusion_on_first_n_qubits(n, total_qubits)

qc = QuantumCircuit(total_qubits)

# Uniform superposition over all subsets.
qc.h(range(n))

N = 2 ** n
M = len(solutions)
num_iterations = max(1, floor((pi / 4) * sqrt(N / M)))

print("Total qubits:", total_qubits)
print("Grover iterations:", num_iterations)

for _ in range(num_iterations):
    qc.append(oracle, range(total_qubits))
    qc.append(diffusion, range(total_qubits))

# Measure only selector bits.
c = ClassicalRegister(n, "c")
qc.add_register(c)
for i in range(n):
    qc.measure(i, c[i])

print(qc.draw(output="text", fold=-1))

In [ ]:
def decode_qiskit_count_key(key: str):
    '''
    Qiskit count strings display highest classical bit on the left.
    We measured x[i] -> c[i], so reverse the string to recover x-order.
    '''
    return tuple(int(bit) for bit in key.replace(" ", "")[::-1])


sim = AerSimulator()

# Aer cannot run the custom Oracle_WeightedAdder gate directly.
# Transpile decomposes it into simulator-supported instructions.
compiled_qc = transpile(qc, sim)
result = sim.run(compiled_qc, shots=2000).result()
counts = result.get_counts()

print("Raw counts:")
print(counts)

decoded = {decode_qiskit_count_key(k): v for k, v in counts.items()}
print("\nDecoded counts in x[0],x[1],x[2] order:")
print(decoded)

print("\nExpected marked bitstrings:")
print(solutions)

## What this version is good for

This notebook is good for:

- showing Grover search on Subset Sum,
- avoiding hard-coded solutions,
- keeping the circuit compact.

But it hides the arithmetic structure. It does not visibly show the paper's shadow registers \(b_i\), running sums, or oracle optimization issues.